# Experiment: VAE Exploration

Objective:
- Train the first-pass VAE directly from this notebook.
- Inspect convergence, decoded samples, and saved run artifacts in one workflow.
- Keep the model code in `convoy_sim/vae.py`, but make the training workflow notebook-first.


## Workflow

This notebook is the active VAE training path.

Top-to-bottom flow:
1. set config and hyperparameters
2. run training from the notebook
3. load the generated run artifacts
4. inspect loss curves and decoded samples


In [10]:
from __future__ import annotations

import csv
import json
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch.optim import Adam
from torch.utils.data import DataLoader

from convoy_sim.vae import AttackProfileVAE, build_vae_datasets, vae_loss
from convoy_sim.workflows import ensure_dir, git_sha, resolve_run_dir, write_json, write_yaml

PROJECT_ROOT = Path.cwd()
PROJECT_ROOT


PosixPath('/Users/matthewplambeck/Desktop/Convoy Layout Project')

In [11]:
# Config: adjust here before running the notebook
TRAIN_PATH = Path("data/attack_profiles/synthetic/train_random_v2.jsonl")
VALID_PATH = Path("data/attack_profiles/synthetic/valid_random_v2.jsonl")
OUTPUT_ROOT = Path("results/runs")
RUN_NAME = "notebook"

SEED = 1945
EPOCHS = 50
BATCH_SIZE = 128
LEARNING_RATE = 1e-3
BETA = 0.05
LATENT_DIM = 4
HIDDEN_DIM = 32
DEVICE = "auto"
NUM_WORKERS = 0
SAMPLE_COUNT = 16

# Training control
RUN_TRAINING = True
EXISTING_RUN_DIR: str | None = None


## Helpers

The next cell mirrors the saved-run artifact pattern used elsewhere in the repo,
but keeps the training loop directly inside the notebook workflow.


In [12]:
def resolve_device(requested: str) -> str:
    if requested != "auto":
        return requested
    if torch.cuda.is_available():
        return "cuda"
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return "mps"
    return "cpu"

def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def mean_stats(stats_list: list[dict[str, float]]) -> dict[str, float]:
    if not stats_list:
        return {"loss": 0.0, "recon_loss": 0.0, "kl_loss": 0.0, "beta": 0.0}
    keys = stats_list[0].keys()
    return {key: float(np.mean([item[key] for item in stats_list], dtype=float)) for key in keys}

def train_one_epoch(model, data_loader, optimizer, device: str, beta: float) -> dict[str, float]:
    model.train()
    batch_stats = []
    for batch in data_loader:
        batch = batch.to(device)
        optimizer.zero_grad(set_to_none=True)
        recon, mu, logvar = model(batch)
        loss, stats = vae_loss(recon, batch, mu, logvar, beta=beta)
        loss.backward()
        optimizer.step()
        batch_stats.append(stats)
    return mean_stats(batch_stats)

def eval_one_epoch(model, data_loader, device: str, beta: float) -> dict[str, float]:
    model.eval()
    batch_stats = []
    with torch.no_grad():
        for batch in data_loader:
            batch = batch.to(device)
            recon, mu, logvar = model(batch)
            _, stats = vae_loss(recon, batch, mu, logvar, beta=beta)
            batch_stats.append(stats)
    return mean_stats(batch_stats)

def write_history_csv(path: Path, history: list[dict[str, float]]) -> None:
    ensure_dir(path.parent)
    fieldnames = [
        "epoch",
        "train_loss",
        "train_recon_loss",
        "train_kl_loss",
        "valid_loss",
        "valid_recon_loss",
        "valid_kl_loss",
    ]
    with path.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        for row in history:
            writer.writerow(row)

def run_training_notebook() -> Path:
    overall_start = time.perf_counter()
    seed_everything(int(SEED))
    device = resolve_device(DEVICE)
    output_root = PROJECT_ROOT / OUTPUT_ROOT
    run_dir = resolve_run_dir(output_root, "vae", RUN_NAME)
    checkpoint_dir = run_dir / "checkpoints"
    ensure_dir(checkpoint_dir)

    data_start = time.perf_counter()
    train_ds, valid_ds, preprocessor = build_vae_datasets(
        train_path=PROJECT_ROOT / TRAIN_PATH,
        valid_path=PROJECT_ROOT / VALID_PATH,
    )
    data_seconds = time.perf_counter() - data_start

    generator = torch.Generator().manual_seed(int(SEED))
    train_loader = DataLoader(
        train_ds,
        batch_size=int(BATCH_SIZE),
        shuffle=True,
        generator=generator,
        num_workers=int(NUM_WORKERS),
    )
    valid_loader = DataLoader(
        valid_ds,
        batch_size=int(BATCH_SIZE),
        shuffle=False,
        num_workers=int(NUM_WORKERS),
    )

    model = AttackProfileVAE(latent_dim=int(LATENT_DIM), hidden_dim=int(HIDDEN_DIM)).to(device)
    optimizer = Adam(model.parameters(), lr=float(LEARNING_RATE))

    history = []
    best_valid_loss = float("inf")
    best_epoch = 0
    train_start = time.perf_counter()

    for epoch in range(1, int(EPOCHS) + 1):
        train_stats = train_one_epoch(model, train_loader, optimizer, device, float(BETA))
        valid_stats = eval_one_epoch(model, valid_loader, device, float(BETA))
        row = {
            "epoch": int(epoch),
            "train_loss": float(train_stats["loss"]),
            "train_recon_loss": float(train_stats["recon_loss"]),
            "train_kl_loss": float(train_stats["kl_loss"]),
            "valid_loss": float(valid_stats["loss"]),
            "valid_recon_loss": float(valid_stats["recon_loss"]),
            "valid_kl_loss": float(valid_stats["kl_loss"]),
        }
        history.append(row)
        best_marker = ""
        if row["valid_loss"] < best_valid_loss:
            best_valid_loss = row["valid_loss"]
            best_epoch = int(epoch)
            best_marker = " *best"
            torch.save({
                "epoch": best_epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "preprocessor": preprocessor.to_dict(),
                "hyperparameters": {
                    "latent_dim": int(LATENT_DIM),
                    "hidden_dim": int(HIDDEN_DIM),
                    "learning_rate": float(LEARNING_RATE),
                    "batch_size": int(BATCH_SIZE),
                    "beta": float(BETA),
                },
            }, checkpoint_dir / "model_best.pt")
        print(
            f"epoch {epoch:03d} | "
            f"train_loss={row['train_loss']:.4f} "
            f"train_recon={row['train_recon_loss']:.4f} "
            f"train_kl={row['train_kl_loss']:.4f} | "
            f"valid_loss={row['valid_loss']:.4f} "
            f"valid_recon={row['valid_recon_loss']:.4f} "
            f"valid_kl={row['valid_kl_loss']:.4f}"
            f"{best_marker}"
        )

    training_seconds = time.perf_counter() - train_start
    torch.save({
        "epoch": int(EPOCHS),
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "preprocessor": preprocessor.to_dict(),
        "hyperparameters": {
            "latent_dim": int(LATENT_DIM),
            "hidden_dim": int(HIDDEN_DIM),
            "learning_rate": float(LEARNING_RATE),
            "batch_size": int(BATCH_SIZE),
            "beta": float(BETA),
        },
    }, checkpoint_dir / "model_latest.pt")

    sample_start = time.perf_counter()
    model.eval()
    with torch.no_grad():
        decoded = model.sample(int(SAMPLE_COUNT), device=torch.device(device)).cpu().numpy()
    sample_payload = [
        preprocessor.decode_profile_fields(
            decoded[idx],
            profile_id=f"VAE_SAMPLE_{idx + 1:04d}",
            name=f"vae_sample_{idx + 1:04d}",
        )
        for idx in range(int(SAMPLE_COUNT))
    ]
    sample_seconds = time.perf_counter() - sample_start

    write_history_csv(run_dir / "training_history.csv", history)
    write_json(run_dir / "sampled_profiles.json", {"profiles": sample_payload})

    final_train = history[-1]
    best_row = history[best_epoch - 1]
    metrics_summary = {
        "dataset": {
            "train_samples": int(len(train_ds)),
            "valid_samples": int(len(valid_ds)),
            "input_dim": int(model.input_dim),
            "feature_names": list(preprocessor.feature_names),
        },
        "model": {
            "latent_dim": int(model.latent_dim),
            "hidden_dim": int(model.hidden_dim),
            "parameter_count": int(sum(int(p.numel()) for p in model.parameters())),
            "beta": float(BETA),
        },
        "training": {
            "epochs": int(EPOCHS),
            "batch_size": int(BATCH_SIZE),
            "learning_rate": float(LEARNING_RATE),
            "best_epoch": int(best_epoch),
            "final_train_loss": float(final_train["train_loss"]),
            "final_train_recon_loss": float(final_train["train_recon_loss"]),
            "final_train_kl_loss": float(final_train["train_kl_loss"]),
            "final_valid_loss": float(final_train["valid_loss"]),
            "final_valid_recon_loss": float(final_train["valid_recon_loss"]),
            "final_valid_kl_loss": float(final_train["valid_kl_loss"]),
            "best_valid_loss": float(best_row["valid_loss"]),
            "best_valid_recon_loss": float(best_row["valid_recon_loss"]),
            "best_valid_kl_loss": float(best_row["valid_kl_loss"]),
        },
        "samples": {
            "count": int(SAMPLE_COUNT),
            "output_file": "sampled_profiles.json",
        },
        "timing": {
            "dataset_load_seconds": float(data_seconds),
            "training_seconds": float(training_seconds),
            "sample_decode_seconds": float(sample_seconds),
            "total_seconds": float(time.perf_counter() - overall_start),
        },
    }
    manifest = {
        "workflow": "vae_train_notebook",
        "git_sha": git_sha(PROJECT_ROOT),
        "device": device,
        "seed": int(SEED),
        "train_path": str(TRAIN_PATH),
        "valid_path": str(VALID_PATH),
        "output_root": str(OUTPUT_ROOT),
        "run_name": str(RUN_NAME),
        "hyperparameters": {
            "epochs": int(EPOCHS),
            "batch_size": int(BATCH_SIZE),
            "learning_rate": float(LEARNING_RATE),
            "beta": float(BETA),
            "latent_dim": int(LATENT_DIM),
            "hidden_dim": int(HIDDEN_DIM),
            "sample_count": int(SAMPLE_COUNT),
        },
        "preprocessor": preprocessor.to_dict(),
        "artifacts": {
            "history_csv": "training_history.csv",
            "metrics_summary_json": "metrics_summary.json",
            "run_manifest_json": "run_manifest.json",
            "best_checkpoint": "checkpoints/model_best.pt",
            "latest_checkpoint": "checkpoints/model_latest.pt",
            "sampled_profiles_json": "sampled_profiles.json",
        },
    }
    resolved_cfg = {
        "dataset": {
            "train_path": str(TRAIN_PATH),
            "valid_path": str(VALID_PATH),
        },
        "training": {
            "epochs": int(EPOCHS),
            "batch_size": int(BATCH_SIZE),
            "learning_rate": float(LEARNING_RATE),
            "beta": float(BETA),
            "latent_dim": int(LATENT_DIM),
            "hidden_dim": int(HIDDEN_DIM),
            "device": device,
            "num_workers": int(NUM_WORKERS),
            "sample_count": int(SAMPLE_COUNT),
            "seed": int(SEED),
        },
    }

    write_yaml(run_dir / "config_resolved.yaml", resolved_cfg)
    write_json(run_dir / "metrics_summary.json", metrics_summary)
    write_json(run_dir / "run_manifest.json", manifest)
    return run_dir


## Train Or Load Existing Run

- If `RUN_TRAINING = True`, this cell launches a fresh run and stores its artifacts.
- If `RUN_TRAINING = False`, set `EXISTING_RUN_DIR` above and this cell will inspect that run instead.


In [13]:
if RUN_TRAINING:
    RUN_DIR = run_training_notebook()
else:
    if not EXISTING_RUN_DIR:
        raise ValueError("Set EXISTING_RUN_DIR when RUN_TRAINING is False")
    RUN_DIR = PROJECT_ROOT / EXISTING_RUN_DIR

RUN_DIR


PosixPath('/Users/matthewplambeck/Desktop/Convoy Layout Project/results/runs/vae/20260428_144653_notebook')

In [14]:
METRICS_PATH = RUN_DIR / "metrics_summary.json"
MANIFEST_PATH = RUN_DIR / "run_manifest.json"
HISTORY_PATH = RUN_DIR / "training_history.csv"
SAMPLES_PATH = RUN_DIR / "sampled_profiles.json"

metrics = json.loads(METRICS_PATH.read_text(encoding="utf-8"))
manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
history = pd.read_csv(HISTORY_PATH)
samples = json.loads(SAMPLES_PATH.read_text(encoding="utf-8"))["profiles"]

summary = {
    "train_samples": metrics["dataset"]["train_samples"],
    "valid_samples": metrics["dataset"]["valid_samples"],
    "latent_dim": metrics["model"]["latent_dim"],
    "hidden_dim": metrics["model"]["hidden_dim"],
    "parameter_count": metrics["model"]["parameter_count"],
    "beta": metrics["model"]["beta"],
    "best_epoch": metrics["training"]["best_epoch"],
    "best_valid_loss": metrics["training"]["best_valid_loss"],
    "final_valid_loss": metrics["training"]["final_valid_loss"],
    "device": manifest["device"],
    "total_seconds": metrics["timing"]["total_seconds"],
}
summary


{'train_samples': 5000,
 'valid_samples': 1000,
 'latent_dim': 4,
 'hidden_dim': 32,
 'parameter_count': 3088,
 'beta': 0.05,
 'best_epoch': 50,
 'best_valid_loss': 0.40725259855389595,
 'final_valid_loss': 0.40725259855389595,
 'device': 'mps',
 'total_seconds': 16.670527959009632}

In [15]:
history.tail()


,epoch,train_loss,train_recon_loss,train_kl_loss,valid_loss,valid_recon_loss,valid_kl_loss
45,46,0.409288,0.160290,4.979958,0.409891,0.164754,4.902743
46,47,0.410279,0.162011,4.965367,0.410746,0.161923,4.976459
47,48,0.406209,0.157710,4.969995,0.412689,0.164664,4.960498
48,49,0.409128,0.160378,4.974986,0.411643,0.161197,5.008915
49,50,0.406861,0.156772,5.001795,0.407253,0.162013,4.904783


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
history.plot(x="epoch", y=["train_loss", "valid_loss"], ax=axes[0], title="Total loss")
history.plot(x="epoch", y=["train_recon_loss", "valid_recon_loss"], ax=axes[1], title="Reconstruction loss")
axes[0].grid(True, alpha=0.3)
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
history.plot(x="epoch", y=["train_kl_loss", "valid_kl_loss"], ax=ax, title="KL loss")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## Decoded Sample Inspection

These are latent-prior samples decoded directly through the VAE and saved with the run.
They are not yet passed through post-decode feasibility + audit gating.


In [ ]:
samples_df = pd.DataFrame(samples)
samples_df[["profile_id", "u_pos", "base_bearing_rad", "spread_rad", "launch_delay_s", "salvo_interval_s", "u_boat_initial_speed_mps"]].head(10)


In [ ]:
samples_df[["spread_rad", "launch_delay_s", "salvo_interval_s", "u_boat_initial_speed_mps"]].describe()


## Next Steps

- Add checkpoint reload cells so you can sample from `model_best.pt` without retraining.
- Add post-decode feasibility + audit filtering for VAE-generated profiles.
- Compare generated-sample distributions against the source synthetic dataset audit.
